In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

## Load

In [2]:
df = pd.read_csv('C:\\Users\\ASUS\\OneDrive\\Desktop\\IITB_internship\\Final_submission\\EEG_IVT_EYE_final_merged_data.csv')

In [3]:
df.head()

,QuestionKey,Delta_TP9_mean,Delta_TP9_median,Delta_TP9_std,Delta_TP9_min,Delta_TP9_max,Delta_AF7_mean,Delta_AF7_median,Delta_AF7_std,Delta_AF7_min,...,ET_DistanceRight_min,ET_DistanceRight_max,FixationCount,SaccadeCount,AvgFixationDuration,AvgSaccadeAmplitude,TotalScanpathLength,MeanGazeVelocity,MeanGazeAcceleration,StdGazeVelocity
0,1Item1,0.827583,0.883827,0.188228,0.239536,1.063340,0.614266,0.722867,0.465170,-0.316656,...,-1.0,581.469910,23,23,446.953215,6.616173,635.152638,24.672133,-215.323812,57.101757
1,1Item10,1.068161,1.119733,0.299992,0.270186,1.488891,0.605479,0.590025,0.304866,-0.328473,...,-1.0,582.609497,63,81,322.570735,8.103427,2698.441257,31.186753,-75.258382,64.929459
2,1Item2,0.960843,0.976382,0.276460,0.327823,1.408459,0.365654,0.396421,0.443385,-0.484325,...,-1.0,581.804138,44,45,417.909081,5.050328,1005.015288,23.677622,-148.160480,51.597881
3,1Item3,0.882369,0.844921,0.232593,0.488667,1.464429,0.468913,0.587266,0.353725,-0.351208,...,-1.0,581.955933,58,66,276.852200,5.174702,1324.723712,26.220196,-146.531625,56.289097
4,1Item4,0.804393,0.746569,0.257972,0.438703,1.517063,0.538407,0.627426,0.315776,-0.258268,...,-1.0,580.769897,59,60,330.590026,5.673720,1424.103776,25.169221,-145.018022,51.763097


In [4]:
df.shape

(1364, 132)

In [5]:
TARGET = 'ResponseTime'
DROP_COLS = ['QuestionKey', 'Participant', TARGET, 'Category'] 


In [6]:
df.dropna(subset=[TARGET], inplace=True)

In [7]:
X = df.drop(columns=DROP_COLS, errors='ignore')
y = df[TARGET]

## Clean 

In [8]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.median(), inplace=True)

## Split and Scale

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# feature scaling

In [10]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
# Convert scaled arrays back to DataFrames with column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)


- Feature Engineering  - Selection og top 40  features using RandomForestRegressor

In [12]:
selector_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
selector_model.fit(X_train_scaled, y_train)
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': selector_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

In [13]:
# Select the top 40 features
N_FEATURES_TO_SELECT = 40
top_features = importances['Feature'].head(N_FEATURES_TO_SELECT).tolist()

In [14]:
print(top_features)

['FixationCount', 'SaccadeCount', 'ET_GazeRighty_mean', 'Beta_TP9_min', 'Delta_AF8_min', 'Beta_TP10_min', 'Alpha_AF8_min', 'Delta_TP9_min', 'ET_GazeLefty_mean', 'Alpha_TP10_min', 'Theta_TP10_max', 'Theta_TP9_min', 'Theta_TP10_min', 'ET_PupilRight_mean', 'AvgFixationDuration', 'ET_GazeRightx_max', 'ET_GazeRightx_median', 'ET_DistanceLeft_std', 'Theta_AF8_min', 'Alpha_TP9_max', 'Theta_AF7_min', 'ET_PupilRight_std', 'Alpha_TP9_min', 'Theta_AF7_max', 'Delta_TP10_min', 'ET_DistanceRight_mean', 'Delta_AF7_min', 'ET_PupilRight_max', 'Alpha_AF7_min', 'ET_GazeRightx_std', 'ET_DistanceRight_std', 'ET_GazeLefty_max', 'ET_GazeRightx_mean', 'ET_GazeRighty_median', 'MeanGazeAcceleration', 'Delta_AF8_std', 'Alpha_TP10_max', 'Alpha_AF8_max', 'ET_PupilLeft_mean', 'Delta_TP9_std']


In [15]:
# Filter the datasets to include only the top features
X_train_selected = X_train_scaled[top_features]
X_test_selected = X_test_scaled[top_features]

# Hyperparameter Tuning 

In [16]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_samples_leaf': [20, 30, 40],
    'subsample': [0.8, 0.9, 1.0]
}

## Randomized Search with 5-fold cross-validation

In [17]:
random_search = RandomizedSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=20, 
    cv=5, 
    n_jobs=-1,
    scoring='r2',
    random_state=42,
    verbose=1
)

In [18]:
random_search.fit(X_train_selected, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=5, estimator=GradientBoostingRegressor(random_state=42),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [3, 5, 7],
                                        'min_samples_leaf': [20, 30, 40],
                                        'n_estimators': [100, 200, 300],
                                        'subsample': [0.8, 0.9, 1.0]},
                   random_state=42, scoring='r2', verbose=1)

In [19]:
print("\nBest parameters found:", random_search.best_params_)



Best parameters found: {'subsample': 1.0, 'n_estimators': 300, 'min_samples_leaf': 20, 'max_depth': 3, 'learning_rate': 0.1}


In [20]:
final_model = random_search.best_estimator_
y_pred = final_model.predict(X_test_selected)

final_r2 = r2_score(y_test, y_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Model Performance")
print(f"R-squared (R²): {final_r2:.4f}")
print(f"Root Mean Squared Error (RMSE): {final_rmse:.4f} seconds")

Model Performance
R-squared (R²): 0.9325
Root Mean Squared Error (RMSE): 3.4850 seconds


- trying to use different Models to evaluate performnace 

In [21]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR

In [22]:
TOP_40_FEATURES = [
    'FixationCount', 'SaccadeCount', 'ET_GazeRighty_mean', 'Beta_TP9_min', 
    'Delta_AF8_min', 'Beta_TP10_min', 'Alpha_AF8_min', 'Delta_TP9_min', 
    'ET_GazeLefty_mean', 'Alpha_TP10_min', 'Theta_TP10_max', 'Theta_TP9_min', 
    'Theta_TP10_min', 'ET_PupilRight_mean', 'AvgFixationDuration', 'ET_GazeRightx_max', 
    'ET_GazeRightx_median', 'ET_DistanceLeft_std', 'Theta_AF8_min', 'Alpha_TP9_max', 
    'Theta_AF7_min', 'ET_PupilRight_std', 'Alpha_TP9_min', 'Theta_AF7_max', 
    'Delta_TP10_min', 'ET_DistanceRight_mean', 'Delta_AF7_min', 'ET_PupilRight_max', 
    'Alpha_AF7_min', 'ET_GazeRightx_std', 'ET_DistanceRight_std', 'ET_GazeLefty_max', 
    'ET_GazeRightx_mean', 'ET_GazeRighty_median', 'MeanGazeAcceleration', 'Delta_AF8_std', 
    'Alpha_TP10_max', 'Alpha_AF8_max', 'ET_PupilLeft_mean', 'Delta_TP9_std'
]


In [23]:
X_selected = X[TOP_40_FEATURES]

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [25]:
models = {
    "XGBoost": XGBRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Support Vector Machine": SVR(),
    "Ridge Regression": Ridge()
}


In [26]:
results = {}
print("\n--- Training and Evaluating All Models ---")
for name, model in models.items():
    print(f"Training {name}...")
    # Train the model
    model.fit(X_train_scaled, y_train)
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    # Calculate performance
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'R-squared': r2, 'RMSE': rmse}


--- Training and Evaluating All Models ---
Training XGBoost...
Training Gradient Boosting...
Training Random Forest...
Training Support Vector Machine...
Training Ridge Regression...


In [27]:

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by='R-squared', ascending=False)

print("\n--- Final Model Performance Leaderboard ---")
print(results_df)

best_model_name = results_df.index[0]
print(f"\nThe best performing model is: {best_model_name}")


--- Final Model Performance Leaderboard ---
                        R-squared      RMSE
Gradient Boosting        0.933552  3.458362
Random Forest            0.927740  3.606444
XGBoost                  0.923987  3.698908
Ridge Regression         0.880733  4.633300
Support Vector Machine   0.739163  6.851961

The best performing model is: Gradient Boosting


- In previous we tried only taking the top 40 features but here we tried taking all the features 

In [28]:
# try after taking all the features
df = pd.read_csv('C:\\Users\\ASUS\\OneDrive\\Desktop\\IITB_internship\\Final_submission\\EEG_IVT_EYE_final_merged_data.csv')

In [29]:
TARGET = 'ResponseTime'
df.dropna(subset=[TARGET], inplace=True)

# Using ALL features this time
X = df.drop(columns=['QuestionKey', 'Participant', TARGET, 'Category'], errors='ignore')
y = df[TARGET]

X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.median(), inplace=True)
print(f"Training on all {X.shape[1]} features.")

Training on all 128 features.


In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [31]:
models = {
    "XGBoost": XGBRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Support Vector Machine": SVR(),
    "Ridge Regression": Ridge()
}

In [32]:
results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    results[name] = {'R-squared': r2, 'RMSE': rmse}

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by='R-squared', ascending=False)


print(results_df)

best_model_name = results_df.index[0]
print(f"\n The best performing model with all features is: {best_model_name}")

Training XGBoost...
Training Gradient Boosting...
Training Random Forest...
Training Support Vector Machine...
Training Ridge Regression...
                        R-squared       RMSE
XGBoost                  0.937412   3.356402
Gradient Boosting        0.933649   3.455843
Random Forest            0.920092   3.792495
Ridge Regression         0.898185   4.280907
Support Vector Machine   0.421054  10.208198

 The best performing model with all features is: XGBoost


In [35]:

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


final_model = GradientBoostingRegressor(random_state=42)
final_model.fit(X_train_scaled, y_train)
predictions = final_model.predict(X_test_scaled)
results_df = df.loc[y_test.index].copy()

results_df['Actual_ResponseTime'] = y_test
results_df['Predicted_ResponseTime'] = predictions

output_df = results_df[['QuestionKey', 'Participant', 'Actual_ResponseTime', 'Predicted_ResponseTime']]

output_df.to_csv('test_set_predictions.csv', index=False)

print(output_df)

     QuestionKey  Participant  Actual_ResponseTime  Predicted_ResponseTime
428       3Item7           12            14.737640               14.699710
930       2Item2           27            12.460655               11.793035
781       2Item8           22            26.588983               26.380340
451       2Item7           13             6.474266                5.152477
429       3Item8           12            18.051560               18.373099
...          ...          ...                  ...                     ...
755       3Item6           21             4.216803                3.429110
526       2Item2           15            24.965431               26.590631
243       1Item5            8            10.755557               11.174017
358       3Item4           10             5.686144                8.192838
1118      1Item8           32            65.570507               63.520552

[273 rows x 4 columns]


1. When using the 40 features subset, the best performing model was Gradient Boosting regressor .
2. When using the 128 features subset, the best performing model was XGBoost.
3. our final model will be XGBoost regressor 